# Generacion de Embeddings InternVL2-2B

**OpenGVLab/InternVL2-2B** — LLM multimodal hibrido (Shanghai AI Lab, 2024)

## Diferencias clave vs los otros 3 modelos

| Modelo | Paradigma | Espacio compartido text/img |
|---|---|---|
| CLIP | Contrastivo dual | Si |
| BLIP | Contrastivo + ITM | Si |
| Qwen2-VL | LLM generativo | No |
| **InternVL2** | **LLM generativo hibrido** | **No** |

InternVL2 tiene un **InternViT** (vision encoder) que en versiones anteriores se entreno contrastivamente, lo que puede hacer que sus embeddings de imagen sean mejores que los de Qwen2-VL.

## Estrategia de extraccion

Como en Qwen2-VL: forward pass + mean pooling sobre los hidden states de la ultima capa del LLM.

## Tiempo estimado

T4 GPU: ~30-45 minutos totales.

## Importante

InternVL2 requiere `trust_remote_code=True` porque tiene codigo Python personalizado en su repositorio de HuggingFace. Es el repo oficial de OpenGVLab.

## Pasos

1. **Runtime → Change runtime type → GPU (T4 o mejor)**
2. **Runtime → Run all**
3. Descargar `internvl_embeddings.zip` al final
4. En tu PC, descomprimir en `clustering-python/embeddings/`

In [ ]:
# CELDA 1: Instalar dependencias
!pip install -q --upgrade datasets huggingface_hub
!pip install -q transformers torchvision scikit-learn pillow accelerate timm einops sentencepiece

In [ ]:
# CELDA 2: Imports y verificacion de GPU
import os, time, zipfile
from pathlib import Path
import numpy as np
import torch
import torchvision.transforms as T
from torchvision.transforms.functional import InterpolationMode
from PIL import Image
from transformers import AutoModel, AutoTokenizer

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)
if device == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))
    print(f'VRAM disponible: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')
else:
    print('ADVERTENCIA: Sin GPU este modelo NO funcionara razonablemente.')

In [ ]:
# CELDA 3: Cargar modelo InternVL2-2B
# Requiere trust_remote_code=True porque InternVL2 tiene codigo custom
MODEL_NAME = 'OpenGVLab/InternVL2-2B'
print('Cargando', MODEL_NAME, '...')
print('(Primera vez tarda ~3-5 min descargando ~5 GB)')

model = AutoModel.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    trust_remote_code=True,
    device_map='auto',
).eval()

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    use_fast=False,
)

# Encontrar la dim del LLM
if hasattr(model.config, 'llm_config'):
    EMBED_DIM = model.config.llm_config.hidden_size
elif hasattr(model.config, 'text_config'):
    EMBED_DIM = model.config.text_config.hidden_size
else:
    EMBED_DIM = 2048  # default para InternVL2-2B

print(f'Modelo listo. Dimension de embedding: {EMBED_DIM}')
print(f'VRAM ocupada: {torch.cuda.memory_allocated() / 1024**3:.2f} GB')

In [ ]:
# CELDA 4: Preprocesamiento de imagenes para InternVL2
# InternVL2 usa una normalizacion especifica con tamaño 448x448

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)
INPUT_SIZE = 448

def build_image_transform():
    """Transformacion estandar de InternVL2: resize a 448x448 + normalizacion ImageNet."""
    return T.Compose([
        T.Lambda(lambda img: img.convert('RGB') if img.mode != 'RGB' else img),
        T.Resize((INPUT_SIZE, INPUT_SIZE), interpolation=InterpolationMode.BICUBIC),
        T.ToTensor(),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ])

image_transform = build_image_transform()
print('Transformacion de imagen lista (448x448 + ImageNet norm)')

In [ ]:
# CELDA 5: Funciones de embedding
# Estrategia: forward pass + mean pooling sobre los hidden states del LLM

@torch.inference_mode()
def embed_texts(texts, batch_size=8, max_length=512):
    """
    Embeddings de texto con InternVL2.
    
    Pasamos solo texto al LLM (sin imagen) y hacemos mean pooling sobre los
    hidden states de la ultima capa.
    
    Devuelve (N, EMBED_DIM) float32 L2-normalizados.
    """
    all_embs = []
    total = len(texts)
    
    # Acceder al LLM interno (en InternVL2 esta en model.language_model)
    llm = model.language_model
    
    for i in range(0, total, batch_size):
        batch = list(texts[i:i+batch_size])
        
        # Tokenizar con padding y truncado
        inputs = tokenizer(
            batch,
            return_tensors='pt',
            padding=True,
            truncation=True,
            max_length=max_length,
        ).to(device)
        
        # Forward pass al LLM solicitando hidden states
        outputs = llm(
            **inputs,
            output_hidden_states=True,
            return_dict=True,
        )
        
        # Hidden states de la ultima capa: (batch, seq_len, hidden_size)
        last_hidden = outputs.hidden_states[-1]
        
        # Mean pooling enmascarado (ignorar padding)
        mask = inputs.attention_mask.unsqueeze(-1).float()
        summed = (last_hidden * mask).sum(dim=1)
        counts = mask.sum(dim=1).clamp(min=1)
        emb = summed / counts
        
        # L2 normalizar
        emb = torch.nn.functional.normalize(emb, p=2, dim=-1)
        
        all_embs.append(emb.float().cpu().numpy())
        
        if (i // batch_size) % 5 == 0:
            print(f'  {min(i+batch_size, total)}/{total}', end='\r')
    
    return np.concatenate(all_embs, axis=0)


@torch.inference_mode()
def embed_images(images, batch_size=4):
    """
    Embeddings de imagen con InternVL2.
    
    Pasamos las imagenes por el vision encoder (InternViT) y hacemos mean
    pooling sobre los hidden states del patch tokens.
    
    Devuelve (N, vision_dim) float32 L2-normalizados.
    """
    all_embs = []
    total = len(images)
    
    # Acceder al vision encoder (InternViT)
    vision_model = model.vision_model
    
    for i in range(0, total, batch_size):
        batch = images[i:i+batch_size]
        
        # Preprocesar imagenes
        pixel_values = torch.stack([image_transform(img) for img in batch])
        pixel_values = pixel_values.to(device).to(torch.float16)
        
        # Forward pass al vision encoder
        vision_out = vision_model(
            pixel_values=pixel_values,
            output_hidden_states=False,
            return_dict=True,
        )
        
        # last_hidden_state: (batch, num_patches, hidden_dim)
        last_hidden = vision_out.last_hidden_state
        
        # Mean pooling sobre todos los patch tokens
        # (no hay attention mask aqui, todos los patches son validos)
        emb = last_hidden.mean(dim=1)
        
        # L2 normalizar
        emb = torch.nn.functional.normalize(emb, p=2, dim=-1)
        
        all_embs.append(emb.float().cpu().numpy())
        
        if (i // batch_size) % 5 == 0:
            print(f'  {min(i+batch_size, total)}/{total} imagenes', end='\r')
        
        # Liberar memoria periodicamente
        if i % 100 == 0:
            torch.cuda.empty_cache()
    
    return np.concatenate(all_embs, axis=0)

In [ ]:
# CELDA 6: Helper para guardar .npz
OUT_DIR = Path('embeddings/internvl')
OUT_DIR.mkdir(parents=True, exist_ok=True)

def save_npz(name, X, y, class_names, source):
    """Guarda embeddings como .npz con metadata."""
    out_path = OUT_DIR / (name + '.npz')
    metadata = {
        'source': source, 'model': MODEL_NAME,
        'dim': int(X.shape[1]), 'n': int(X.shape[0]),
        'n_classes': int(len(np.unique(y))),
    }
    np.savez_compressed(
        out_path,
        X=X.astype(np.float32),
        y=np.asarray(y, dtype=np.int64),
        class_names=np.asarray(class_names, dtype=object),
        metadata=np.asarray([metadata], dtype=object),
    )
    size_mb = out_path.stat().st_size / 1024**2
    norms = np.linalg.norm(X, axis=1)
    norm_ok = np.allclose(norms, 1.0, atol=1e-4)
    print(f'  [OK] {name:28s} N={metadata["n"]:5d} D={metadata["dim"]:4d} K={metadata["n_classes"]:3d} {size_mb:.2f}MB L2={"OK" if norm_ok else "FALLO"}')

## TEXTO - Datasets 1 a 5

In [ ]:
# [1/8] 20 Newsgroups (5 clases)
from sklearn.datasets import fetch_20newsgroups

print('\n[1/8] 20 Newsgroups (5 clases)')
t0 = time.time()
cats_5 = [
    'rec.sport.hockey',
    'rec.sport.baseball',
    'sci.med',
    'sci.space',
    'talk.politics.misc',
]
ds = fetch_20newsgroups(
    subset='all', categories=cats_5,
    remove=('headers', 'footers', 'quotes')
)
texts = ds.data
labels = list(ds.target)
class_names = list(ds.target_names)
print(f'  Distribucion: {dict(zip(class_names, [labels.count(i) for i in range(len(class_names))]))}')
X = embed_texts(texts)
save_npz('20ng_5classes', X, labels, class_names, 'sklearn:fetch_20newsgroups (5 cats)')
print(f'  Tiempo: {time.time()-t0:.1f}s')

In [ ]:
# [2/8] BBC News
from datasets import load_dataset, concatenate_datasets

print('\n[2/8] BBC News')
t0 = time.time()
ds_train = load_dataset('SetFit/bbc-news', split='train')
try:
    ds_test = load_dataset('SetFit/bbc-news', split='test')
    ds = concatenate_datasets([ds_train, ds_test])
except Exception:
    ds = ds_train

texts = list(ds['text'])
labels = list(ds['label'])
if 'label_text' in ds.column_names:
    label_to_text = {}
    for lid, ltxt in zip(ds['label'], ds['label_text']):
        label_to_text[lid] = ltxt
    class_names = [label_to_text[i] for i in sorted(label_to_text.keys())]
else:
    class_names = [str(c) for c in sorted(set(labels))]
print(f'  Clases: {class_names}')
X = embed_texts(texts)
save_npz('bbc_news', X, labels, class_names, 'huggingface:SetFit/bbc-news')
print(f'  Tiempo: {time.time()-t0:.1f}s')

In [ ]:
# [3/8] 20 Newsgroups (2 clases)
print('\n[3/8] 20 Newsgroups (2 clases)')
t0 = time.time()
cats_2 = ['alt.atheism', 'soc.religion.christian']
ds = fetch_20newsgroups(
    subset='all', categories=cats_2,
    remove=('headers', 'footers', 'quotes')
)
texts, labels = ds.data, list(ds.target)
class_names = list(ds.target_names)
X = embed_texts(texts)
save_npz('20ng_2classes', X, labels, class_names, 'sklearn:fetch_20newsgroups')
print(f'  Tiempo: {time.time()-t0:.1f}s')

In [ ]:
# [4/8] 20 Newsgroups (3 clases)
print('\n[4/8] 20 Newsgroups (3 clases)')
t0 = time.time()
cats_3 = ['comp.graphics', 'rec.sport.hockey', 'sci.med']
ds = fetch_20newsgroups(
    subset='all', categories=cats_3,
    remove=('headers', 'footers', 'quotes')
)
texts, labels = ds.data, list(ds.target)
class_names = list(ds.target_names)
X = embed_texts(texts)
save_npz('20ng_3classes', X, labels, class_names, 'sklearn:fetch_20newsgroups')
print(f'  Tiempo: {time.time()-t0:.1f}s')

In [ ]:
# [5/8] AG News (subsample 1000/clase = 4000 total)
print('\n[5/8] AG News (4k subsample)')
t0 = time.time()
ds_ag = load_dataset('fancyzhx/ag_news', split='train')
ds_ag = ds_ag.shuffle(seed=42)

PER_CLASS = 1000
selected_idx = []
counts = {0: 0, 1: 0, 2: 0, 3: 0}
for i, label in enumerate(ds_ag['label']):
    if counts[label] < PER_CLASS:
        selected_idx.append(i)
        counts[label] += 1
    if all(c >= PER_CLASS for c in counts.values()):
        break

ds_sub = ds_ag.select(selected_idx)
texts = list(ds_sub['text'])
labels = list(ds_sub['label'])
class_names = ['World', 'Sports', 'Business', 'Sci/Tech']
print(f'  Distribucion: {dict(zip(class_names, [labels.count(i) for i in range(4)]))}')
X = embed_texts(texts)
save_npz('ag_news_4k', X, labels, class_names, 'huggingface:fancyzhx/ag_news (1k/class)')
print(f'  Tiempo: {time.time()-t0:.1f}s')

## IMAGENES - Datasets 6 a 8

⚠️ InternVL2 procesa imagenes con resolucion 448x448 (mayor que CLIP 224x224). Los embeddings de imagen vienen del **InternViT** (vision encoder), no del LLM completo, por lo que la dim sera diferente a la de texto.

In [ ]:
# [6/8] ORL Faces (Olivetti) - 400 imagenes
from sklearn.datasets import fetch_olivetti_faces

print('\n[6/8] ORL Faces')
t0 = time.time()
ds_oli = fetch_olivetti_faces(shuffle=False)

images = []
for img_arr in ds_oli.images:
    img = (img_arr * 255).astype(np.uint8)
    pil = Image.fromarray(img, mode='L').convert('RGB')
    images.append(pil)

labels = list(ds_oli.target)
class_names = [f'person_{i}' for i in range(40)]
X = embed_images(images)
save_npz('orl_faces', X, labels, class_names, 'sklearn:fetch_olivetti_faces')
print(f'  Tiempo: {time.time()-t0:.1f}s')

torch.cuda.empty_cache()

In [ ]:
# [7/8] MNIST (subsample 500/clase = 5000 total)
import torchvision

print('\n[7/8] MNIST (5k subsample)')
t0 = time.time()
mnist = torchvision.datasets.MNIST(
    root='/tmp/mnist', train=True, download=True
)

PER_CLASS = 500
targets = np.asarray(mnist.targets)
selected_idx = []
for c in range(10):
    idx_c = np.where(targets == c)[0][:PER_CLASS]
    selected_idx.extend(idx_c.tolist())

images, labels = [], []
for i in selected_idx:
    img, lab = mnist[i]
    images.append(img.convert('RGB'))
    labels.append(int(lab))

class_names = [str(i) for i in range(10)]
X = embed_images(images)
save_npz('mnist_5k', X, labels, class_names, 'torchvision:MNIST (500/class)')
print(f'  Tiempo: {time.time()-t0:.1f}s')

torch.cuda.empty_cache()

In [ ]:
# [8/8] Fashion-MNIST (subsample 500/clase = 5000 total)
print('\n[8/8] Fashion-MNIST (5k subsample)')
t0 = time.time()
fmnist = torchvision.datasets.FashionMNIST(
    root='/tmp/fmnist', train=True, download=True
)

PER_CLASS = 500
targets = np.asarray(fmnist.targets)
selected_idx = []
for c in range(10):
    idx_c = np.where(targets == c)[0][:PER_CLASS]
    selected_idx.extend(idx_c.tolist())

images, labels = [], []
for i in selected_idx:
    img, lab = fmnist[i]
    images.append(img.convert('RGB'))
    labels.append(int(lab))

class_names = ['T-shirt', 'Trouser', 'Pullover', 'Dress', 'Coat',
              'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle_boot']
X = embed_images(images)
save_npz('fashion_mnist_5k', X, labels, class_names, 'torchvision:FashionMNIST (500/class)')
print(f'  Tiempo: {time.time()-t0:.1f}s')

torch.cuda.empty_cache()

## Resumen y descarga

In [ ]:
# Resumen de todo lo generado
print('\n=== RESUMEN FINAL ===')
print(f'{"Dataset":<28} {"N":>6} {"D":>5} {"K":>4} {"MB":>7} {"L2":>4}')
print('-' * 60)
total_mb = 0
all_ok = True
for f in sorted(OUT_DIR.glob('*.npz')):
    size_mb = f.stat().st_size / 1024**2
    total_mb += size_mb
    npz = np.load(f, allow_pickle=True)
    X, y = npz['X'], npz['y']
    norms = np.linalg.norm(X, axis=1)
    norm_ok = np.allclose(norms, 1.0, atol=1e-4)
    if not norm_ok:
        all_ok = False
    print(f'{f.stem:<28} {X.shape[0]:>6} {X.shape[1]:>5} {len(np.unique(y)):>4} {size_mb:>7.2f} {"OK" if norm_ok else "FALLO":>4}')

print('-' * 60)
print(f'{"TOTAL":28} {"":>6} {"":>5} {"":>4} {total_mb:>7.2f}')
print()
if all_ok:
    print('✓ Todos los archivos pasaron verificacion L2.')
else:
    print('⚠ Algunos archivos tienen problemas de normalizacion.')

In [ ]:
# Crear ZIP para descargar
ZIP_PATH = 'internvl_embeddings.zip'
with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in OUT_DIR.glob('*.npz'):
        zf.write(f, arcname=f'embeddings/internvl/{f.name}')

zip_mb = os.path.getsize(ZIP_PATH) / 1024**2
print(f'ZIP creado: {ZIP_PATH} ({zip_mb:.2f} MB)')

In [ ]:
# Descargar al navegador
from google.colab import files
files.download(ZIP_PATH)

In [ ]:
# (OPCIONAL) Guardar en Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# import shutil
# shutil.copy(ZIP_PATH, '/content/drive/MyDrive/internvl_embeddings.zip')
# print('Copiado a Drive')

## Pasos siguientes en tu PC

1. Descomprimir `internvl_embeddings.zip` en la raiz del proyecto `clustering-python/`
   - Quedara la estructura: `embeddings/internvl/<dataset>.npz`

2. Verificar:
   ```cmd
   venv\Scripts\activate
   python embeddings_loader.py internvl
   ```

3. Correr los 8 algoritmos con InternVL2:
   ```cmd
   python testing.py 9 --model internvl
   ```

4. Generar tabla comparativa con todos los modelos:
   ```cmd
   python build_results_table.py --all
   ```

## Notas para tu tesis

- **InternVL2 es generativo** (no contrastivo), igual que Qwen2-VL.
- Los embeddings de **texto** vienen del LLM (Internlm2) via mean pooling.
- Los embeddings de **imagen** vienen del **InternViT** (vision encoder), que en versiones anteriores se entreno contrastivamente. Esto puede dar mejores resultados de clustering de imagenes que Qwen2-VL.
- Es importante notar que **las dim de texto e imagen pueden ser diferentes** (texto del LLM ~2048, imagen del InternViT ~1024). No los compares directamente entre modalidades.
- Resolucion de imagen: 448x448 (mayor que CLIP/BLIP 224x224).